[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L08.ipynb)

# Lecture 08 — Tensor Fluency Lab & Gradients, Intuitively

*Twelve shape puzzles to close Part I — then the one idea that makes learning work.*

**PyTorch from Ground Up · Volume I · Chapters 12–13** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Chain three or four tensor operations without hesitating — and predict every intermediate shape.
- Solve the checkerboard, attention-reshape, distance-matrix and one-hot puzzles without a Python loop.
- Explain a gradient as the slope of the loss with respect to a parameter — no calculus required.
- Apply the update rule `new = old − lr × gradient` by hand.
- Predict what a too-large or too-small learning rate does.
- Re-read the Lecture 03 training loop and say what lines ❹ and ❺ really do.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Knowing every operation is not fluency

You’ve met every tensor operation in Part I. **Fluency** is chaining three or four without hesitating. Knowing them one at a time is necessary; fluency only comes from practice. This lab is that practice.

<sub>Source: lecture slides 3–5</sub>

## 02 · Warm-up puzzles 1–3



<sub>Source: lecture slides 6–8</sub>

## 03 · Medium puzzles 4–8



<sub>Source: lecture slides 9–13</sub>

## 04 · Hard puzzles 9–12 and the capstones

Puzzle 10 — the **attention-style reshape**: four operations, one chain. Start with (2, 10, 256) — batch, sequence, features. Split the features into 4 heads of 64, move heads before sequence, then flatten the last two axes.

<sub>Source: lecture slides 14–23</sub>

**`puzzle_09.py`** — predict the output, then run.

In [ ]:
# Puzzle 9 — Checkerboard
i = torch.arange(8)
board = ((i[:, None] + i[None, :]) % 2 == 0).long()
print(board)

<details><summary>▸ Expected output</summary>

```
tensor([[1, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 1, 0, 1],
        [1, 0, 1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 1, 0, 1]])
```
</details>

**`puzzle_11.py`** — predict the output, then run.

In [ ]:
# Puzzle 11 — Batch distance matrix
pts = torch.randn(5, 3)
diff = pts.unsqueeze(1) - pts.unsqueeze(0)    # (5,1,3)-(1,5,3)→(5,5,3)
dist = (diff ** 2).sum(dim=2).sqrt()          # (5, 5)
print(dist.shape)
print(dist[0, 0])        # 0 — distance to self

<details><summary>▸ Expected output</summary>

```
torch.Size([5, 5])
tensor(0.)
```
</details>

## 05 · The hill and the slope

In Lecture 03 the model learned. But **how did it know which way to move its numbers?** The answer is one idea — the gradient — built here from scratch, no calculus required.

<sub>Source: lecture slides 25–31</sub>

**`slope_check.py`** — predict the output, then run.

In [ ]:
# Verify the slope of y = x² at x = 3
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print(x.grad)           # tensor(6.) — slope = 2x = 6

<details><summary>▸ Expected output</summary>

```
tensor(6.)
```
</details>

## 06 · Stepping downhill

The minus sign is the whole trick: the gradient points **uphill**, so subtracting it moves you downhill. The learning rate sets the size of each step.

<sub>Source: lecture slides 32–37</sub>

**`ten_steps.py`** — predict the output, then run.

In [ ]:
w = 3.0                    # starting position
lr = 0.1                   # learning rate
for step in range(10):
    grad = 2 * w           # slope of w²
    w = w - lr * grad      # step downhill
    print(f"w={w:.4f}   loss={w**2:.4f}")

<details><summary>▸ Expected output</summary>

```
w=2.4000   loss=5.7600
w=1.9200   loss=3.6864
w=1.5360   loss=2.3593
w=1.2288   loss=1.5099
w=0.9830   loss=0.9664
w=0.7864   loss=0.6185
w=0.6291   loss=0.3958
w=0.5033   loss=0.2533
w=0.4027   loss=0.1621
w=0.3221   loss=0.1038
```
</details>

🧪 *Interactive lab in the app:* **Gradient descent on y = w²** — try it there to see this idea move.

## 07 · The learning rate — and Lecture 03 with new eyes

Lines ❹ and ❺ **are this whole lecture**. ❹ computes the slope for every parameter; ❺ subtracts lr × slope from each. Gradient descent in a loop — feeling the slope and stepping downhill, two hundred times.

<sub>Source: lecture slides 38–42</sub>

**`too_large.py`** — predict the output, then run.

In [ ]:
# Too large — overshoots
w, lr = 3.0, 1.5
for _ in range(5):
    w = w - lr * (2 * w)
    print(f"w={w:.1f}   loss={w**2:.1f}")

<details><summary>▸ Expected output</summary>

```
w=-6.0   loss=36.0
w=12.0   loss=144.0
w=-24.0   loss=576.0
w=48.0   loss=2304.0
w=-96.0   loss=9216.0
```
</details>

> ⚠️ **Watch out — nan or inf? Your learning rate is too high**  
> If the loss suddenly jumps to `nan` or `inf` during training, the learning rate is almost certainly too high. Reduce it by a factor of 10 and try again.
>
> This is the most common cause of `nan` or `inf` in training — and the first thing to reduce when it happens.

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
x = torch.linspace(0, 2, 20).unsqueeze(1)
y = 3 * x + 1 + 0.3 * torch.randn_like(x)
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)


**`training_loop.py`** — predict the output, then run.

In [ ]:
pred = model(x)              # ❶ compute prediction
loss = loss_fn(pred, y)      # ❷ measure error
optimizer.zero_grad()        # ❸ clear old slopes
loss.backward()              # ❹ compute the gradient
optimizer.step()             # ❺ step downhill

## The rule — worth remembering

**Part I — you now have tensor fluency**

1. Creation, indexing, slicing, reshaping, squeezing, permuting, broadcasting, reductions, joining, splitting — every one is now assumed.
2. Chaining them is the skill: predict each shape before you run.
3. Broadcasting replaces loops: unsqueeze to line the axes up, then operate.

**Chapter 13 — gradients**

1. A gradient is the slope of the loss with respect to a parameter.
2. A negative gradient means “the loss decreases if you increase the parameter.” A positive gradient means “the loss decreases if you decrease it.”
3. Gradient descent: new = old − lr × gradient. The minus sign turns “uphill” into “downhill.”
4. The learning rate controls step size. Too big → overshoot. Too small → stall.
5. For many parameters, the gradient is a vector — one slope per parameter. PyTorch computes the whole vector with `.backward()`.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Try it — change the start and the rate

Run the 10-step gradient descent on w². Then change the starting value to `w = -5.0`. Does it still find the minimum? Now try `lr = 0.01` with 10 steps — is it close to zero? How many steps does it need?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

From −5 it still converges: the gradient there is negative, so subtracting it moves w up toward 0. With lr = 0.01 each step multiplies w by 0.98, so after 10 steps w ≈ 2.45 — nowhere near zero. Reaching w ≈ 0.3 takes about 114 steps instead of 10. Try both in the lab.
</details>

### Exercise 02 · Autograd

Compute the gradient of y = x² at x = 5. What do you get? Confirm it equals 2x.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`x = torch.tensor(5.0, requires_grad=True); (x ** 2).backward(); x.grad` → `tensor(10.)` = 2 × 5.
</details>

### Exercise 03 · A moved minimum

Run gradient descent on y = (x − 4)² from x = 0 with lr = 0.1 for 30 steps. What value does x converge to? Why?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

It converges to 4 (after 30 steps x ≈ 3.995). The slope 2(x − 4) is zero only at x = 4 — the bottom of the curve moved, and gradient descent follows it.
</details>

### Exercise 04 · Learning-rate sweep

Run y = x² with lr = 0.01, 0.1, 0.5, 1.1, 20 steps each. Which converges fastest? Which diverges?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

lr = 0.5 lands on the minimum in a single step (w − 0.5 × 2w = 0). lr = 0.1 converges steadily, lr = 0.01 crawls, and lr = 1.1 diverges — each step multiplies w by −1.2.
</details>

### Exercise 05 · Two parameters

Minimize y = a² + b² from a = 3, b = 4, printing both each step. Do they both approach zero?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Yes. The gradient is the vector (2a, 2b); applying the same update rule to each shrinks both by the same factor every step.
</details>

### Exercise 06 · One sentence

Why does the training loop call `zero_grad()` before `backward()`? What would happen if you skipped it?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Because gradients accumulate: without zeroing, each `backward()` adds to the previous gradients, so the steps are built from stale slopes and the model diverges.
</details>

### Interview-style

“In plain language, what is a gradient and how does a neural network use it to learn?” Three sentences, no formulas.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

A gradient is the slope of the error with respect to one adjustable number in the model: it says whether nudging that number up would make the error larger or smaller, and how steeply. A network computes this slope for every one of its numbers at once, then moves each a small step in the downhill direction. Repeating that — measure the slope, step downhill — thousands of times is what training is.
</details>